<a href="https://colab.research.google.com/github/jaggu1367/data-quality-inspector/blob/demo/GE_Demo2_SQL_negation_approach.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Install packages if needed
!pip install -q great_expectations pyspark

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 90.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 813.6/813.6 kB 49.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 118.1/118.1 kB 7.5 MB/s eta 0:00:00


In [2]:
import sys
print(sys.version)

3.12.12 (main, Oct 10 2025, 08:52:57) [GCC 11.4.0]


In [3]:
import great_expectations as gx
print(gx.__version__)

1.11.3


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [4]:
## the below code is expected to be run only once per runtime

import great_expectations as gx
from pyspark.sql import SparkSession

# Optional. Request a File Data Context from a specific folder.
context = gx.get_context(mode="file", project_root_dir="./ge")

# Define the Data Source name
data_source_name = "temp_data_source_name_1"
# Add the Data Source to the Data Context
data_source = context.data_sources.add_spark(data_source_name)

# Define the Data Asset name
data_asset_name = "temp_dataframe_data_asset"

# Add a Data Asset to the Data Source
data_asset = data_source.add_dataframe_asset(name=data_asset_name)

# Define the Batch Definition name
batch_definition_name = "temp_batch_definition"

# Add a Batch Definition to the Data Asset
batch_definition = data_asset.add_batch_definition_whole_dataframe(
    batch_definition_name
)

In [14]:
# Provide a dataframe through Batch Parameters
spark = SparkSession.builder.getOrCreate()
df = spark.createDataFrame([
    {"bonus": 6000, "tenure": 3, "salary": 45000, "department": "Sales", "commission": 1500},
    {"bonus": 8000, "tenure": 1, "salary": 60000, "department": "HR", "commission": 1800},
    {"bonus": 9500, "tenure": 4, "salary": 40000, "department": "Engineering", "commission": 1200},
])
batch_parameters = {"dataframe": df}

# Create an Expectation to test
expectation = gx.expectations.ExpectColumnValuesToBeBetween(
    column="bonus", max_value=10000, min_value=6000
)

# Get the dataframe as a Batch
batch = batch_definition.get_batch(batch_parameters=batch_parameters)

# Test the Expectation
validation_results = batch.validate(expectation)
print(validation_results)

Calculating Metrics:   0%|          | 0/13 [00:00<?, ?it/s]

{
  "success": true,
  "expectation_config": {
    "type": "expect_column_values_to_be_between",
    "kwargs": {
      "batch_id": "temp_data_source_name_1-temp_dataframe_data_asset",
      "column": "bonus",
      "min_value": 6000.0,
      "max_value": 10000.0
    },
    "meta": {},
    "severity": "critical"
  },
  "result": {
    "element_count": 3,
    "unexpected_count": 0,
    "unexpected_percent": 0.0,
    "partial_unexpected_list": [],
    "missing_count": 0,
    "missing_percent": 0.0,
    "unexpected_percent_total": 0.0,
    "unexpected_percent_nonmissing": 0.0,
    "partial_unexpected_counts": []
  },
  "meta": {},
  "exception_info": {
    "raised_exception": false,
    "exception_traceback": null,
    "exception_message": null
  }
}


In [16]:

# SQL_EXPECTATIONS as provided
SQL_EXPECTATIONS = [
    {"expectation": "commission BETWEEN 1000 AND 2000", "filter":""},
    {"expectation": "department IN ('Sales','HR','Engineering')", "filter":""},
    {"expectation": "bonus BETWEEN 5000 AND 10000", "filter":"department = 'Sales'"}
]

# Helper to normalize each SQL expression into a safe WHERE clause
def _wrap_condition(cond: str) -> str:
    # Ensure each top-level expression is parenthesized to avoid precedence issues
    return f"({cond})"

validation_summary = []
for idx, item in enumerate(SQL_EXPECTATIONS, start=1):
    where_clause = _wrap_condition(item['expectation'])
    # Build a full query using the {batch} placeholder your environment expects
    unexpected_rows_query = f"""
        SELECT *
        FROM {{batch}}
        WHERE !({where_clause})
    """

    description = f"Expectation {idx}: rows must satisfy {item['expectation']}"
    expectation_obj = gx.expectations.UnexpectedRowsExpectation(
        unexpected_rows_query=unexpected_rows_query,
        description=description,
    )
    if item['filter']:
        df = df.filter(item['filter'])
    batch_parameters = {"dataframe": df}
    batch = batch_definition.get_batch(batch_parameters=batch_parameters)
    cond = item['expectation'] + (f" with filter: {item['filter']}" if item['filter'] else "")
    # Validate and collect results
    result = batch.validate(expectation_obj)
    validation_summary.append((idx, cond, result.get("success", False), result))

    # Print concise per-expectation outcome
    print(f"Deviation {idx} - condition: {cond}")
    print(f"result: {result.get("result", "[]")}")
    print("  Success:", result.get("success", False))
    print("-" * 60)

# Overall summary
all_passed = all(item[2] for item in validation_summary)
print("Overall validation passed:" , all_passed)

# Optionally raise if any expectation failed (uncomment if desired)
if not all_passed:
    raise AssertionError("One or more SQL expectations failed.")

Calculating Metrics:   0%|          | 0/2 [00:00<?, ?it/s]

Deviation 1 - condition: commission BETWEEN 1000 AND 2000
result: {'observed_value': 0}
  Success: True
------------------------------------------------------------


Calculating Metrics:   0%|          | 0/2 [00:00<?, ?it/s]

Deviation 2 - condition: department IN ('Sales','HR','Engineering')
result: {'observed_value': 0}
  Success: True
------------------------------------------------------------


Calculating Metrics:   0%|          | 0/2 [00:00<?, ?it/s]

Deviation 3 - condition: bonus BETWEEN 5000 AND 10000 with filter: department = 'Sales'
result: {'observed_value': 0}
  Success: True
------------------------------------------------------------
Overall validation passed: True
